# Quantization Evaluation Analysis

This notebook is for one question:

> After GRPO post-training changes a model, does that change survive 4-bit quantization?

The active result is now the **clean Day 5 data1000 chat-template matrix** for Qwen2.5-1.5B-Instruct. The old raw-prompt 1.5B matrix is kept only as an audit artifact because it was measured through a stopping/truncation bug.

The final clean 1.5B eval has **six cases**. Read these labels literally:

| File label | Plain label | Meaning |
| --- | --- | --- |
| `base_fp16` | Base FP16 | original Qwen2.5-1.5B-Instruct, no GRPO, no 4-bit quantization |
| `g8_dr100_fp16` | GRPO FP16 | the 1.5B checkpoint after data1000 chat-format GRPO, no 4-bit quantization |
| `base_w4` | Base bnb-W4 | original 1.5B loaded with bitsandbytes NF4 4-bit quantization |
| `g8_dr100_w4` | GRPO bnb-W4 | data1000 GRPO 1.5B loaded with bitsandbytes NF4 4-bit quantization |
| `base_awq` | Base AWQ-W4 | original 1.5B quantized with AWQ W4G128 using chat-formatted calibration |
| `g8_dr100_awq` | GRPO AWQ-W4 | data1000 GRPO 1.5B quantized with AWQ W4G128 using chat-formatted calibration |

The 0.5B runs are **not the final claim**. They are shown because they explain why we moved to 1.5B: 0.5B is too close to the 4-bit quantization floor.


## What The Quantization Words Mean

**FP16/BF16** means no 4-bit compression. It is the reference behavior.

**bnb-NF4 W4** means bitsandbytes load-time 4-bit quantization. It uses a fixed NF4 codebook and converts weights while loading the model. It is a useful cheap probe, but it does not use a calibration dataset from this task.

**AWQ W4G128** means Activation-aware Weight Quantization: 4-bit weights, group size 128. AWQ is **calibration-aware** because it first runs representative calibration text through the model, measures activation scales, and uses those measurements to decide which weight channels need better protection/scaling before quantization.

In this repo, the AWQ calibration text is GSM8K-formatted prompt + reference solution text. So AWQ is closer to the real scheme we care about; bnb-W4 is the cheap comparison.

Important caveat: calibration-aware does **not** mean AWQ must have higher exact-match accuracy than bnb-NF4 on every finite downstream eval. Absolute bnb-vs-AWQ accuracy depends on implementation details, calibration text, group size, decoding, and sample noise. The research comparison is mainly within each scheme: Base vs GRPO and the resulting gain-survival.

## Metrics And How To Read Them

Do not only look at one raw accuracy number. The research quantities are:

```text
delta_fp16 = accuracy(GRPO FP16) - accuracy(Base FP16)
delta_w4   = accuracy(GRPO bnb-W4) - accuracy(Base bnb-W4)
delta_awq  = accuracy(GRPO AWQ-W4) - accuracy(Base AWQ-W4)
```

Then:

```text
gain_survival_w4  = delta_w4  - delta_fp16
gain_survival_awq = delta_awq - delta_fp16
```

If `gain_survival_awq` is near zero, the GRPO gain survived AWQ. If it is strongly negative, AWQ erased or reversed the GRPO gain.

The code cells below are intentionally annotated. Most cells only load CSV/JSON files from finished Slurm jobs; they do **not** rerun model inference.

In [ ]:
from pathlib import Path
import os
import json
import math

from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

PQS_ROOT = PQS_ROOT
REPO_ROOT = PQS_ROOT / 'repos/posttrain-quant-serve'
LOG_DIR = REPO_ROOT / 'logs'
OUTLIER_DIR_0P5 = PQS_ROOT / 'diagnostics/weight_outliers_0p5b_g8_dr100'
OUTLIER_DIR_1P5 = PQS_ROOT / 'diagnostics/weight_outliers_1p5b_g8_dr100'

WEIGHT_DIAGNOSTIC_SPECS = {
    '0p5_g8_dr100': {
        'title': '0.5B base vs GRPO g8_dr100',
        'model_size': '0.5B',
        'path': OUTLIER_DIR_0P5,
        'command': '''OUTPUT_DIR=$PQS_ROOT/diagnostics/weight_outliers_0p5b_g8_dr100 \
MODEL_SPECS="base=Qwen/Qwen2.5-0.5B-Instruct g8_dr100=$PQS_ROOT/ckpts/qwen2_5_0_5b_grpo_g8_dr100" \
sbatch --job-name=pqs-weight-0p5 \
  --account=cavestru0 \
  --time=00:10:00 \
  --export=ALL \
  slurm/weight_outliers.sbatch''',
    },
    '1p5_g8_dr100': {
        'title': '1.5B base vs GRPO g8_dr100',
        'model_size': '1.5B',
        'path': OUTLIER_DIR_1P5,
        'command': '''OUTPUT_DIR=$PQS_ROOT/diagnostics/weight_outliers_1p5b_g8_dr100 \
MODEL_SPECS="base=Qwen/Qwen2.5-1.5B-Instruct g8_dr100=$PQS_ROOT/ckpts/qwen2_5_1_5b_grpo_g8_dr100" \
sbatch --job-name=pqs-weight-1p5 \
  --account=cavestru0 \
  --time=00:20:00 \
  --export=ALL \
  slurm/weight_outliers.sbatch''',
    },
}

EVAL_SPECS = {
    '0p5_bnb_w4_test50': {
        'title': '0.5B bnb-NF4 W4 test50',
        'model_size': '0.5B',
        'role': 'dev floor diagnostic',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test50_g8_dr100_bnb_w4_final',
        'job_id': '51161262',
        'command': '''PRECISIONS=fp16,w4 \
EVAL_SPLIT=test \
EVAL_LIMIT=50 \
EVAL_OUTPUT_DIR=$PQS_ROOT/evals/gsm8k_compare_test50_g8_dr100_bnb_w4_final \
sbatch --job-name=pqs-eval-w4-test50 \
  --account=cavestru0 \
  --time=00:15:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
    '0p5_awq_test50': {
        'title': '0.5B AWQ W4G128 test50',
        'model_size': '0.5B',
        'role': 'dev floor diagnostic',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test50_g8_dr100_awq_w4g128',
        'job_id': '51163938',
        'command': '''PRECISIONS=fp16,awq \
EVAL_SPLIT=test \
EVAL_LIMIT=50 \
EVAL_OUTPUT_DIR=$PQS_ROOT/evals/gsm8k_compare_test50_g8_dr100_awq_w4g128 \
sbatch --job-name=pqs-eval-awq-test50 \
  --account=cavestru0 \
  --time=00:20:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
    '1p5_fp16_w4_awq_test100': {
        'title': '1.5B raw-prompt FP16 + bnb-W4 + AWQ test100',
        'model_size': '1.5B',
        'role': 'raw-prompt stopping-bug diagnostic (not final)',
        'path': PQS_ROOT / 'evals/gsm8k_compare_test100_qwen2_5_1_5b_g8_dr100_fp16_w4_awq',
        'job_id': '51173013',
        'command': '''MODEL_TAG=qwen2_5_1_5b \
PRECISIONS=fp16,w4,awq \
EVAL_SPLIT=test \
EVAL_LIMIT=100 \
EVAL_OUTPUT_DIR=$PQS_ROOT/evals/gsm8k_compare_test100_qwen2_5_1_5b_g8_dr100_fp16_w4_awq \
sbatch --job-name=pqs-eval-1p5-test100 \
  --account=cavestru0 \
  --time=00:45:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
    '1p5_data1000_chat_fp16_w4_awq_test100': {
        'title': '1.5B data1000 chat GRPO FP16 + bnb-W4 + AWQ test100',
        'model_size': '1.5B',
        'role': 'final clean result',
        'path': PQS_ROOT / 'evals/gsm8k_chat_test100_qwen2_5_1_5b_data1000_fp16_w4_awq',
        'job_id': '51224222',
        'command': '''MODEL_TAG=qwen2_5_1_5b \
PRECISIONS=fp16,w4,awq \
EVAL_SPLIT=test \
EVAL_LIMIT=100 \
DENSE_DTYPE=fp16 \
SKIP_REFERENCE_PPL=1 \
TRAINED_MODEL=$PQS_ROOT/ckpts/qwen2_5_1_5b_grpo_data1000_chat \
BASE_AWQ_MODEL=$PQS_ROOT/ckpts_awq/qwen2_5_1_5b_base_awq_w4g128_chatcalib \
TRAINED_AWQ_MODEL=$PQS_ROOT/ckpts_awq/qwen2_5_1_5b_data1000_chat_awq_w4g128 \
EVAL_OUTPUT_DIR=$PQS_ROOT/evals/gsm8k_chat_test100_qwen2_5_1_5b_data1000_fp16_w4_awq \
sbatch --job-name=pqs-eval-data1000-full \
  --account=huterer2 \
  --time=01:30:00 \
  --export=ALL \
  slurm/eval_gsm8k_compare.sbatch''',
    },
}

ACTIVE_EVAL = '1p5_data1000_chat_fp16_w4_awq_test100'

# Human-readable names for the six rows in results_matrix.csv.
CASE_LABELS = {
    'base_fp16': 'Base FP16',
    'g8_dr100_fp16': 'GRPO FP16',
    'base_w4': 'Base bnb-W4',
    'g8_dr100_w4': 'GRPO bnb-W4',
    'base_awq': 'Base AWQ-W4',
    'g8_dr100_awq': 'GRPO AWQ-W4',
}

# Stable order for tables and plots. This avoids confusing alphabetic ordering.
CASE_ORDER = list(CASE_LABELS)

plt.style.use('default')
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.width', 180)

for key, spec in EVAL_SPECS.items():
    print(f"{key}: {spec['path']} exists={spec['path'].exists()}")

## Helper Functions

In [ ]:
def safe_display(df: pd.DataFrame | None, message: str = 'No rows to display.'):
    if df is None or df.empty:
        print(message)
    else:
        display(df)


def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    return pd.DataFrame(json.loads(line) for line in path.read_text().splitlines() if line.strip())


def read_matrix(eval_dir: Path) -> pd.DataFrame:
    path = eval_dir / 'results_matrix.csv'
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def fmt_pct(x):
    return None if pd.isna(x) else f'{100*x:.1f}%'


def metric(summary: dict, key: str):
    value = summary.get(key)
    if value is None:
        return None
    try:
        return float(value)
    except Exception:
        return value


def run_sacct_command(job_id: str) -> str:
    return f'sacct -j {job_id} --format=JobID,JobName%25,State,ExitCode,Elapsed'


def summarize_eval(label: str, spec: dict) -> dict:
    eval_dir = spec['path']
    summary = read_json(eval_dir / 'summary.json')
    matrix = read_matrix(eval_dir)
    row = {
        'eval': label,
        'title': spec['title'],
        'model_size': spec['model_size'],
        'role': spec['role'],
        'status': 'found' if summary and not matrix.empty else 'missing',
        'split': summary.get('split'),
        'limit': summary.get('limit'),
        'precisions': ','.join(summary.get('precisions', [])) if summary else None,
        'delta_fp16': metric(summary, 'delta_fp16'),
        'delta_w4': metric(summary, 'delta_w4'),
        'delta_awq': metric(summary, 'delta_awq'),
        'gain_survival_w4': metric(summary, 'gain_survival_w4'),
        'gain_survival_awq': metric(summary, 'gain_survival_awq'),
        'quant_drop_base_w4': metric(summary, 'quant_drop_base_w4'),
        'quant_drop_g8_w4': metric(summary, 'quant_drop_g8_w4'),
        'quant_drop_base_awq': metric(summary, 'quant_drop_base_awq'),
        'quant_drop_g8_awq': metric(summary, 'quant_drop_g8_awq'),
        'path': str(eval_dir),
        'job_id': spec.get('job_id'),
    }
    if not matrix.empty:
        for label_name in ['base_fp16', 'g8_dr100_fp16', 'base_w4', 'g8_dr100_w4', 'base_awq', 'g8_dr100_awq']:
            hit = matrix[matrix['label'] == label_name]
            row[f'{label_name}_acc'] = float(hit.iloc[0]['accuracy']) if len(hit) else None
    return row


def explain_gain_survival(summary: dict, quant: str) -> str:
    delta_fp16 = summary.get('delta_fp16')
    delta_q = summary.get(f'delta_{quant}')
    survival = summary.get(f'gain_survival_{quant}')
    if delta_fp16 is None or delta_q is None or survival is None:
        return f'Need both FP16 and {quant} rows.'
    if survival >= -0.05:
        return f'{quant}: GRPO gain mostly survives ({survival:+.3f}).'
    return f'{quant}: GRPO gain is erased/reversed ({survival:+.3f}).'


def print_missing_commands(label: str, spec: dict):
    print()
    print(f"Missing or incomplete: {label}")
    if spec.get('job_id'):
        print('Check job:')
        print(run_sacct_command(spec['job_id']))
    print('Run command:')
    print(spec['command'])


def read_weight_layers(diag_dir: Path) -> pd.DataFrame:
    """Load per-layer weight diagnostics for one base-vs-GRPO run."""
    path = diag_dir / 'weight_outlier_layers.csv'
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def summarize_weight_diag(label: str, spec: dict) -> dict:
    """Return one row describing whether a weight diagnostic artifact is available."""
    diag_dir = spec['path']
    layers_path = diag_dir / 'weight_outlier_layers.csv'
    summary_path = diag_dir / 'weight_outlier_summary.json'
    return {
        'diagnostic': label,
        'title': spec['title'],
        'model_size': spec['model_size'],
        'status': 'found' if layers_path.exists() and summary_path.exists() else 'missing',
        'layers_csv': layers_path.exists(),
        'layers_csv_kib': round(layers_path.stat().st_size / 1024, 1) if layers_path.exists() else None,
        'summary_json': summary_path.exists(),
        'summary_json_kib': round(summary_path.stat().st_size / 1024, 1) if summary_path.exists() else None,
        'path': str(diag_dir),
    }


def weight_distribution_delta_table(layers: pd.DataFrame) -> pd.DataFrame:
    """Compare GRPO against base for weight-distribution metrics.

    Positive delta means the GRPO checkpoint has a larger value than the base model
    for the same metric. This is a descriptive diagnostic only; it does not prove
    quantization quality by itself.
    """
    if layers.empty or 'model' not in layers:
        return pd.DataFrame()
    required = {'base', 'g8_dr100'}
    if not required.issubset(set(layers['model'])):
        return pd.DataFrame()

    metric_cols = [
        'max_abs',
        'mean_abs',
        'median_abs',
        'std',
        'rms',
        'kurtosis',
        'abs_outlier_frac',
        'channel_outlier_frac',
        'channel_max_abs_p99',
        'w4_relative_mse',
        'w4_snr_db',
    ]
    rows = []
    for metric_name in metric_cols:
        if metric_name not in layers.columns:
            continue
        values = []
        for model_label in ['base', 'g8_dr100']:
            sub = layers[layers['model'] == model_label]
            metric_values = pd.to_numeric(sub[metric_name], errors='coerce')
            weights = pd.to_numeric(sub.get('numel', pd.Series(index=sub.index, dtype=float)), errors='coerce')
            mask = metric_values.notna() & weights.notna() & (weights > 0)
            if mask.any():
                value = (metric_values[mask] * weights[mask]).sum() / weights[mask].sum()
            else:
                value = metric_values.mean()
            values.append(value)
        base_value, grpo_value = values
        delta = grpo_value - base_value
        rel_delta = delta / abs(base_value) if pd.notna(base_value) and abs(base_value) > 1e-30 else math.nan
        rows.append({
            'metric': metric_name,
            'base': base_value,
            'g8_dr100': grpo_value,
            'delta_grpo_minus_base': delta,
            'relative_delta': rel_delta,
        })
    return pd.DataFrame(rows)


def top_weight_metric_deltas(layers: pd.DataFrame, metric_name: str, n: int = 12) -> pd.DataFrame:
    """Show layers where GRPO changed a metric most relative to base."""
    if layers.empty or metric_name not in layers.columns:
        return pd.DataFrame()
    key_cols = ['module_name', 'module_type', 'shape']
    base = layers[layers['model'] == 'base'][key_cols + [metric_name]].rename(columns={metric_name: 'base'})
    grpo = layers[layers['model'] == 'g8_dr100'][key_cols + [metric_name]].rename(columns={metric_name: 'g8_dr100'})
    merged = base.merge(grpo, on=key_cols, how='inner')
    if merged.empty:
        return pd.DataFrame()
    merged['delta_grpo_minus_base'] = pd.to_numeric(merged['g8_dr100'], errors='coerce') - pd.to_numeric(merged['base'], errors='coerce')
    merged['abs_delta'] = merged['delta_grpo_minus_base'].abs()
    return merged.sort_values('abs_delta', ascending=False).head(n).drop(columns=['abs_delta'])


## 1. Artifact Checklist

This shows which eval directories are ready. If the 1.5B final matrix is missing, first check whether job `51173013` completed or timed out.

In [ ]:
artifact_rows = []
for label, spec in EVAL_SPECS.items():
    eval_dir = spec['path']
    files = {
        'summary_json': eval_dir / 'summary.json',
        'results_matrix': eval_dir / 'results_matrix.csv',
        'paired_quant': eval_dir / 'paired_quant_effect.csv',
    }
    row = {
        'eval': label,
        'title': spec['title'],
        'path': str(eval_dir),
        'job_id': spec.get('job_id'),
    }
    for name, path in files.items():
        row[name] = path.exists()
        row[f'{name}_kib'] = round(path.stat().st_size / 1024, 1) if path.exists() else None
    artifact_rows.append(row)

artifact_df = pd.DataFrame(artifact_rows)
safe_display(artifact_df)

for label, spec in EVAL_SPECS.items():
    summary_path = spec['path'] / 'summary.json'
    matrix_path = spec['path'] / 'results_matrix.csv'
    if not (summary_path.exists() and matrix_path.exists()):
        print_missing_commands(label, spec)

## 2. Cross-Run Scorecard

This is the compact comparison across the 0.5B floor diagnostics, the old raw-prompt 1.5B audit artifact, and the clean Day 5 data1000 chat-template result.


In [ ]:
scorecard = pd.DataFrame([summarize_eval(label, spec) for label, spec in EVAL_SPECS.items()])
ordered_cols = [
    'eval', 'status', 'model_size', 'role', 'split', 'limit', 'precisions',
    'base_fp16_acc', 'g8_dr100_fp16_acc', 'base_w4_acc', 'g8_dr100_w4_acc', 'base_awq_acc', 'g8_dr100_awq_acc',
    'delta_fp16', 'delta_w4', 'delta_awq', 'gain_survival_w4', 'gain_survival_awq',
    'quant_drop_base_w4', 'quant_drop_g8_w4', 'quant_drop_base_awq', 'quant_drop_g8_awq',
    'path'
]
for col in ordered_cols:
    if col not in scorecard.columns:
        scorecard[col] = None
safe_display(scorecard[ordered_cols])

## 3. Active Eval Matrix

By default this loads the clean Day 5 data1000 chat-template 1.5B matrix. Change `ACTIVE_EVAL` in the configuration cell to inspect an older diagnostic run.


In [ ]:
active_spec = EVAL_SPECS[ACTIVE_EVAL]
EVAL_DIR = active_spec['path']
SUMMARY_JSON = EVAL_DIR / 'summary.json'
RESULTS_MATRIX = EVAL_DIR / 'results_matrix.csv'
PAIRED_QUANT = EVAL_DIR / 'paired_quant_effect.csv'

summary = read_json(SUMMARY_JSON)
matrix = read_matrix(EVAL_DIR)

print('ACTIVE_EVAL =', ACTIVE_EVAL)
print('EVAL_DIR =', EVAL_DIR)
print('summary exists:', SUMMARY_JSON.exists())
print('matrix exists:', RESULTS_MATRIX.exists())

if not matrix.empty:
    matrix['accuracy_pct'] = matrix['accuracy'].map(fmt_pct)
    matrix['stderr_pct'] = matrix['accuracy_stderr'].map(fmt_pct)
    score_cols = [
        'variant', 'precision', 'label', 'correct', 'num_examples', 'accuracy', 'accuracy_stderr',
        'parse_rate', 'prompt_leak_rate', 'hit_max_new_tokens_rate', 'ended_with_eos_rate',
        'completion_tokens_mean', 'completion_chars_mean', 'reference_ppl', 'model'
    ]
    score_cols = [col for col in score_cols if col in matrix.columns]
    safe_display(matrix[score_cols])

    if 'hit_max_new_tokens_rate' in matrix.columns and matrix['hit_max_new_tokens_rate'].fillna(0).gt(0).any():
        display(Markdown(
            '**Audit warning:** at least one row hit `max_new_tokens`. If this rate differs by precision, '
            'the accuracy comparison may be measuring truncation behavior instead of model quality. '
            'Rerun with larger `MAX_NEW_TOKENS` before making a claim.'
        ))
else:
    print_missing_commands(ACTIVE_EVAL, active_spec)

headline_cols = ['delta_fp16', 'delta_w4', 'delta_awq', 'gain_survival_w4', 'gain_survival_awq', 'quant_drop_base_w4', 'quant_drop_g8_w4', 'quant_drop_base_awq', 'quant_drop_g8_awq']
headline = {col: summary.get(col) for col in headline_cols}
safe_display(pd.DataFrame([headline]) if summary else pd.DataFrame(), 'No summary metrics loaded.')

## 4. Headline Interpretation

This cell gives the plain-language answer for the active eval. For the clean data1000 run, the main quantities are `delta_fp16`, `delta_w4`, `delta_awq`, and the two gain-survival metrics.


In [ ]:
if not summary or matrix.empty:
    display(Markdown('**No active result yet.** Check the job status or rerun the command printed above.'))
else:
    display(Markdown(f"### {active_spec['title']}"))
    delta_fp16 = summary.get('delta_fp16')
    display(Markdown(f"- FP16 GRPO gain: `{delta_fp16:+.3f}`" if delta_fp16 is not None else '- FP16 GRPO gain: missing'))
    for quant in ['w4', 'awq']:
        display(Markdown(f"- {explain_gain_survival(summary, quant)}"))

    if active_spec['role'] == 'final clean result':
        display(Markdown(
            '**Interpretation:** data1000 chat-format GRPO produced a small held-out FP16 gain. '
            'The bnb-NF4 W4 delta remains positive and close to the FP16 delta, and the AWQ delta is larger on this test100 slice. '
            'Although bnb-W4 has higher absolute accuracy than AWQ in this matrix, that is not a contradiction and not a universal quantizer ranking; '
            'the relevant result is that the GRPO-vs-base gain survives within both tested W4 schemes.'
        ))
    elif 'stopping-bug' in active_spec['role']:
        display(Markdown(
            '**Interpretation:** this run is retained as an audit artifact only. It used the old raw-prompt setup and should not be used for the final quantization claim.'
        ))
    elif active_spec['model_size'] == '0.5B':
        display(Markdown(
            '**Interpretation:** this is a quantization-floor diagnostic. If quantized base accuracy is already near zero, '
            'the run is useful for debugging but not for the final research claim.'
        ))
    else:
        base_awq = scorecard.loc[scorecard['eval'] == ACTIVE_EVAL, 'base_awq_acc'].iloc[0] if 'base_awq_acc' in scorecard else None
        if pd.notna(base_awq) and base_awq < 0.15:
            display(Markdown(
                '**Warning:** even 1.5B base-AWQ is very low here. Treat the result cautiously and inspect predictions before writing the final claim.'
            ))
        else:
            display(Markdown(
                '**Interpretation:** if base quantized accuracy is meaningfully above floor, this run can answer whether GRPO survives W4/AWQ.'
            ))

## 5. Clear Plots For The Active 1.5B Run

The earlier cross-run plot was confusing because it mixed 0.5B and 1.5B on one axis. This block only plots the active run (`ACTIVE_EVAL`), using the six plain labels from the table at the top.

The first plot is raw accuracy. The second plot is the actual research quantity: GRPO gain before quantization and after each quantization method.

In [ ]:
# If the active eval has not finished, there is nothing to plot.
if matrix.empty:
    print('No active matrix loaded; skipping plots.')
else:
    # Copy so plotting columns do not mutate the original dataframe unexpectedly.
    plot_matrix = matrix.copy()

    # Convert file labels like g8_dr100_awq into readable labels like GRPO AWQ-W4.
    plot_matrix['plain_label'] = plot_matrix['label'].map(CASE_LABELS).fillna(plot_matrix['label'])

    # Force a stable order: Base FP16, GRPO FP16, Base bnb-W4, GRPO bnb-W4, Base AWQ, GRPO AWQ.
    order_map = {label: i for i, label in enumerate(CASE_ORDER)}
    plot_matrix['case_order'] = plot_matrix['label'].map(order_map).fillna(999).astype(int)
    plot_matrix = plot_matrix.sort_values('case_order')

    # Base models are gray; GRPO-trained models are blue.
    colors = ['tab:gray' if row['variant'] == 'base' else 'tab:blue' for _, row in plot_matrix.iterrows()]

    # Hatches distinguish precision/quantization method.
    hatches = {'fp16': '', 'w4': '//', 'awq': 'xx'}

    # Plot 1: direct exact-match accuracy for the six cases.
    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(
        plot_matrix['plain_label'],
        plot_matrix['accuracy'],
        yerr=plot_matrix['accuracy_stderr'],
        capsize=4,
        color=colors,
    )

    # Add hatch patterns and numeric accuracy labels above bars.
    for bar, (_, row) in zip(bars, plot_matrix.iterrows()):
        bar.set_hatch(hatches.get(row['precision'], ''))
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{row['accuracy']:.2f}",
            ha='center',
            va='bottom',
            fontsize=9,
        )

    ax.set_title(f"{active_spec['title']}: GSM8K exact-match accuracy")
    ax.set_ylabel('accuracy')
    ax.set_ylim(0, max(0.4, float(plot_matrix['accuracy'].max()) + 0.08))
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

    # Plot 2: GRPO gains and gain-survival metrics from summary.json.
    gain_rows = [
        {'metric': 'GRPO gain in FP16', 'value': summary.get('delta_fp16'), 'formula': 'GRPO FP16 - Base FP16'},
        {'metric': 'GRPO gain after bnb-W4', 'value': summary.get('delta_w4'), 'formula': 'GRPO bnb-W4 - Base bnb-W4'},
        {'metric': 'GRPO gain after AWQ-W4', 'value': summary.get('delta_awq'), 'formula': 'GRPO AWQ-W4 - Base AWQ-W4'},
        {'metric': 'Survival under bnb-W4', 'value': summary.get('gain_survival_w4'), 'formula': 'delta_w4 - delta_fp16'},
        {'metric': 'Survival under AWQ-W4', 'value': summary.get('gain_survival_awq'), 'formula': 'delta_awq - delta_fp16'},
    ]
    gain_df = pd.DataFrame(gain_rows)
    safe_display(gain_df)

    # Remove missing metrics, for example if an eval did not include AWQ.
    gain_plot = gain_df.dropna(subset=['value'])

    fig, ax = plt.subplots(figsize=(11, 4))
    bars = ax.bar(gain_plot['metric'], gain_plot['value'], color=['tab:blue', 'tab:green', 'tab:orange', 'tab:red', 'tab:purple'][:len(gain_plot)])
    ax.axhline(0, color='black', linewidth=1)

    # Annotate each bar with signed delta, e.g. +0.12 or -0.04.
    for bar, value in zip(bars, gain_plot['value']):
        y = value + (0.01 if value >= 0 else -0.025)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y,
            f'{value:+.2f}',
            ha='center',
            va='bottom' if value >= 0 else 'top',
            fontsize=9,
        )

    ax.set_title('GRPO gain and quantization gain-survival')
    ax.set_ylabel('accuracy delta')
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    plt.show()

## 6. Paired Quantization Effects

This shows how many individual GSM8K examples changed when the same checkpoint moved from FP16 to W4 or AWQ.

In [ ]:
paired = pd.read_csv(PAIRED_QUANT) if PAIRED_QUANT.exists() else pd.DataFrame()
safe_display(paired.head(), 'No paired quantization file loaded for active eval.')

if not paired.empty:
    group_cols = ['variant', 'quant_precision', 'quant_change'] if 'quant_precision' in paired.columns else ['variant', 'quant_change']
    counts = paired.groupby(group_cols).size().reset_index(name='count')
    safe_display(counts)

    if 'quant_precision' in paired.columns:
        pivot = counts.pivot_table(index=['variant', 'quant_precision'], columns='quant_change', values='count', fill_value=0)
    else:
        pivot = counts.pivot_table(index='variant', columns='quant_change', values='count', fill_value=0)
    for col in ['improved', 'worsened', 'unchanged']:
        if col not in pivot.columns:
            pivot[col] = 0
    safe_display(pivot[['improved', 'worsened', 'unchanged']])

    ax = pivot[['improved', 'worsened']].plot(kind='bar', figsize=(8, 4), color=['tab:green', 'tab:red'])
    ax.set_title('Examples changed by quantization')
    ax.set_ylabel('count')
    ax.grid(axis='y', alpha=0.25)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Inspect Changed Examples

Use this to see whether quantization changes are real answer changes or parser/format quirks.

In [ ]:
if paired.empty:
    print('No paired rows to inspect.')
else:
    changed = paired[paired['quant_change'] != 'unchanged'].copy()
    inspect_cols = [
        'variant', 'quant_precision', 'index', 'target_answer', 'fp16_answer', 'quant_answer',
        'fp16_correct', 'quant_correct', 'quant_change', 'question'
    ]
    inspect_cols = [col for col in inspect_cols if col in changed.columns]
    safe_display(changed[inspect_cols].sort_values([col for col in ['variant', 'quant_precision', 'quant_change', 'index'] if col in changed.columns]))

## 8. Prediction Drilldown

Set `EXAMPLE_INDEX` to inspect all available completions for one GSM8K item.

In [ ]:
EXAMPLE_INDEX = 1

if matrix.empty:
    print('No active matrix loaded.')
else:
    prediction_labels = matrix['label'].tolist()
    prediction_frames = {label: read_jsonl(EVAL_DIR / f'{label}_predictions.jsonl') for label in prediction_labels}

    for label, frame in prediction_frames.items():
        if frame.empty or 'index' not in frame.columns or EXAMPLE_INDEX not in set(frame['index']):
            continue
        row = frame[frame['index'] == EXAMPLE_INDEX].iloc[0]
        display(Markdown(f"### {label}"))
        print('target:', row.get('target_answer'))
        print('parsed:', row.get('parsed_answer'))
        print('correct:', row.get('exact_match'))
        print('prompt_leak:', row.get('prompt_leak'))
        print()
        print('question:')
        print(row.get('question'))
        print()
        print('completion:')
        print(row.get('completion'))

## 9. Weight Distribution Diagnostics

This is a **weight-space** check: did GRPO create an obvious global shift in raw-weight properties that would make W4 quantization harder?

Note: the existing 1.5B weight diagnostic was run on the earlier `g8_dr100` checkpoint, not the final data1000 checkpoint. It supports the general observation that this GRPO recipe family did not obviously explode global outlier metrics, but a direct data1000 weight diagnostic would be cleaner if the final writeup needs that exact claim.

These metrics are intentionally coarse. They can show large shifts in scale, tail heaviness, outlier rate, or proxy W4 error. They cannot prove that GRPO left the weights unchanged, because behavior can change through small coordinated directional updates that barely move global histograms.

The metrics mean:

| Metric | Meaning | Why it matters |
| --- | --- | --- |
| `max_abs` | largest absolute weight in a layer | very large weights can force a larger quantization scale |
| `mean_abs` / `median_abs` | typical weight magnitude | tells whether the whole distribution shifted |
| `std` / `rms` | spread / energy of the weights | bigger spread can increase quantization error |
| `kurtosis` | tail heaviness; Gaussian is about 3 | higher values mean heavier tails / more extreme weights |
| `abs_outlier_frac` | fraction of weights above `6 * median(abs(weight))` | crude global outlier rate |
| `channel_outlier_frac` | fraction of output channels whose max is above `6 * median(channel max)` | detects channel-level outliers relevant to group quantization |
| `w4_relative_mse` | proxy reconstruction error from simple symmetric group-wise int4 | cheap estimate of W4 quantization difficulty |
| `w4_snr_db` | signal-to-noise ratio of that W4 proxy | higher is better |

Positive deltas below mean **GRPO is larger than base**. For `w4_relative_mse`, positive is worse. For `w4_snr_db`, negative is worse.

In [ ]:
weight_artifact_rows = [summarize_weight_diag(label, spec) for label, spec in WEIGHT_DIAGNOSTIC_SPECS.items()]
weight_artifact_df = pd.DataFrame(weight_artifact_rows)
safe_display(weight_artifact_df)

for label, spec in WEIGHT_DIAGNOSTIC_SPECS.items():
    row = weight_artifact_df[weight_artifact_df['diagnostic'] == label].iloc[0]
    if row['status'] != 'found':
        print(f"\nMissing {label} weight diagnostic. Run from repo root on Great Lakes:\n")
        print(spec['command'])

### Weight Metric Deltas

This table is the coarse answer to: **did GRPO create a large global, quantization-relevant weight-distribution shift?**

For each available diagnostic run, it computes a weighted average across layers/tensors, using number of weights as the weight. That prevents tiny layers from dominating the summary. A near-zero result here means the global summaries are stable; it does **not** mean the GRPO update was functionally zero.

In [ ]:
weight_delta_tables = {}

for label, spec in WEIGHT_DIAGNOSTIC_SPECS.items():
    layers = read_weight_layers(spec['path'])
    delta_table = weight_distribution_delta_table(layers)
    weight_delta_tables[label] = delta_table
    display(Markdown(f"### {spec['title']}"))
    if delta_table.empty:
        print('No layer table loaded yet.')
    else:
        pretty = delta_table.copy()
        for col in ['base', 'g8_dr100', 'delta_grpo_minus_base', 'relative_delta']:
            pretty[col] = pd.to_numeric(pretty[col], errors='coerce')
        safe_display(pretty.round(6))

        # Short automatic interpretation. This is a coarse quantization-difficulty
        # diagnostic, not a test of whether training changed the model function.
        interesting = pretty[pretty['relative_delta'].abs() > 0.05]
        if interesting.empty:
            display(Markdown(
                '**Interpretation:** no coarse global quantization metric moved by more than 5%. '
                'This does not mean GRPO left the weights unchanged. It means the behaviorally useful update was not visible as a large shift in marginal weight scale, outlier fraction, or simple W4 reconstruction-error summaries.'
            ))
        else:
            display(Markdown('**Interpretation:** at least one global quantization metric moved by more than 5%; inspect the layer-level deltas below.'))

### Layer-Level Changes

A global average can hide one bad layer. These tables show the layers with the largest GRPO-vs-base changes in the main tail/quantization metrics.

In [ ]:
for label, spec in WEIGHT_DIAGNOSTIC_SPECS.items():
    layers = read_weight_layers(spec['path'])
    if layers.empty:
        continue
    display(Markdown(f"### {spec['title']}: largest layer changes"))
    for metric_name in ['max_abs', 'kurtosis', 'channel_outlier_frac', 'w4_relative_mse']:
        display(Markdown(f"**{metric_name}**"))
        safe_display(top_weight_metric_deltas(layers, metric_name).round(6), f'No {metric_name} deltas available.')

### Weight Diagnostic Plots

These plots are descriptive: they show whether GRPO systematically increases tail heaviness or W4 proxy error layer-by-layer. A tight cloud around the diagonal means base and GRPO are almost identical for that metric.

In [ ]:
for label, spec in WEIGHT_DIAGNOSTIC_SPECS.items():
    layers = read_weight_layers(spec['path'])
    if layers.empty:
        continue
    key_cols = ['module_name', 'module_type', 'shape']
    base = layers[layers['model'] == 'base']
    grpo = layers[layers['model'] == 'g8_dr100']
    merged = base[key_cols + ['max_abs', 'kurtosis', 'w4_relative_mse']].merge(
        grpo[key_cols + ['max_abs', 'kurtosis', 'w4_relative_mse']],
        on=key_cols,
        suffixes=('_base', '_g8_dr100'),
    )
    if merged.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric_name in zip(axes, ['max_abs', 'kurtosis', 'w4_relative_mse']):
        x = pd.to_numeric(merged[f'{metric_name}_base'], errors='coerce')
        y = pd.to_numeric(merged[f'{metric_name}_g8_dr100'], errors='coerce')
        mask = x.notna() & y.notna()
        ax.scatter(x[mask], y[mask], s=12, alpha=0.7)
        if mask.any():
            lo = min(x[mask].min(), y[mask].min())
            hi = max(x[mask].max(), y[mask].max())
            ax.plot([lo, hi], [lo, hi], color='black', linewidth=1, alpha=0.6)
        ax.set_title(metric_name)
        ax.set_xlabel('base')
        ax.set_ylabel('GRPO g8_dr100')
        ax.grid(True, alpha=0.25)
    fig.suptitle(spec['title'])
    plt.tight_layout()
    plt.show()

## 10. Current Conclusion

This section summarizes the clean Day 5 matrix. It deliberately treats the old raw-prompt matrix as an audit artifact and uses the data1000 chat-template run as the active result.


In [ ]:
final_label = '1p5_data1000_chat_fp16_w4_awq_test100'
final_row = scorecard[scorecard['eval'] == final_label]

if final_row.empty or final_row.iloc[0]['status'] != 'found':
    display(Markdown('**Final data1000 1.5B matrix is not loaded yet.** Run or check the job printed in the artifact checklist.'))
else:
    r = final_row.iloc[0]
    display(Markdown(
        f"**Clean Day 5 result:** FP16 gain `{r['delta_fp16']:+.3f}`, "
        f"bnb-W4 gain `{r['delta_w4']:+.3f}`, AWQ gain `{r['delta_awq']:+.3f}`. "
        f"Gain survival: bnb-W4 `{r['gain_survival_w4']:+.3f}`, AWQ `{r['gain_survival_awq']:+.3f}`."
    ))

    display(Markdown(
        '**Claim direction:** data1000 GRPO produced a small held-out FP16 gain, and that gain survived the tested W4 quantization schemes. '
        'bnb-NF4 nearly preserved the FP16 gain; AWQ was more favorable to the GRPO checkpoint than to the base checkpoint on this test100 slice. '
        'The fact that bnb-W4 has higher absolute accuracy than AWQ here is an empirical result for this setup, not a general claim that bnb-W4 is the better quantizer. '
        'Do not overclaim beyond test100, but this is the first clean matrix supporting the project story.'
    ))

    stopping_cols = ['label', 'parse_rate', 'prompt_leak_rate', 'hit_max_new_tokens_rate', 'ended_with_eos_rate']
    if not matrix.empty and all(col in matrix.columns for col in stopping_cols):
        display(Markdown('**Stopping sanity check for the final matrix:**'))
        safe_display(matrix[stopping_cols])